# FLUKA: residual nuclei in pPb

Generate pPb events with FLUKA and histogram the mass-number distribution $A$ of residual/fragment nuclei in the final state.

Residual nuclei are identified by PDG nuclear codes (`|pid| > 1e9`, format `10LZZZAAAI`). Free nucleons (`2212`, `2112`) are excluded, so the histogram reflects bound fragments only ($A \geq 2$).

Requires: matplotlib, tqdm, boost-histogram, particle.

In [ ]:
import os
if not os.environ.get("FLUPRO"):
    raise RuntimeError("Set the FLUPRO environment variable to your FLUKA installation path")


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import boost_histogram as bh
from tqdm import tqdm
import numpy as np

from chromo.constants import GeV, TeV
from chromo.kinematics import FixedTarget
from chromo.models.fluka import Fluka

In [ ]:
# p + Pb208 at 1 TeV lab kinetic energy.
ekin = FixedTarget(1 * TeV, "proton", "Pb208")

n_events = 2000

# A-axis: one bin per integer mass number from A=1 to A=210.
a_axis = bh.axis.Regular(210, 0.5, 210.5)

In [ ]:
gen = Fluka(ekin, seed=1)

hA = bh.Histogram(a_axis)

for event in tqdm(gen(n_events), total=n_events):
    fs = event.final_state()
    pid = np.abs(fs.pid.astype(np.int64))
    nuc = pid >= 1_000_000_000
    if not np.any(nuc):
        continue
    A = (pid[nuc] // 10) % 1000
    hA.fill(A)

In [ ]:
fig, (ax_full, ax_heavy) = plt.subplots(1, 2, figsize=(12, 4.5))

a = hA.axes[0]
vals = hA.values() / n_events

for ax in (ax_full, ax_heavy):
    ax.stairs(vals, a.edges, label=gen.label)
    ax.set_xlabel("mass number A of residual nucleus")
    ax.set_ylabel("residuals / event / $\\Delta A$")
    ax.set_yscale("log")
    ax.legend()

ax_full.set_title("Full A range")
ax_full.set_xlim(0, 210)

ax_heavy.set_title("Heavy-residual region (near target A=208)")
ax_heavy.set_xlim(150, 210)

fig.suptitle(f"pPb @ {ekin.elab/1e3:.0f} TeV lab - residual-nucleus mass spectrum")
fig.tight_layout();

In [ ]:
vals = hA.values()
centers = hA.axes[0].centers
total = vals.sum()
if total == 0:
    print("no residual nuclei recorded")
else:
    mean_A = (centers * vals).sum() / total
    max_A = centers[np.nonzero(vals)[0][-1]]
    per_event = total / n_events
    print(f"<A>={mean_A:6.2f}  A_max={max_A:6.1f}  residuals/event={per_event:.3f}")